## Loading dataset

In [1]:
import pandas as pd

df = pd.read_csv("cleaned_reviews.csv.xls.csv")

# Check data
print(df.head())

# Make sure column exists
print(df.columns)

                     review,sentiment,cleaned_review
0  One of the other reviewers has mentioned that ...
1  A wonderful little production. <br /><br />The...
2  I thought this was a wonderful way to spend ti...
3  Basically there's a family where a little boy ...
4  Petter Mattei's "Love in the Time of Money" is...
Index(['review,sentiment,cleaned_review'], dtype='object')


## Preparing data

In [2]:
import pandas as pd

df = pd.read_csv("cleaned_reviews.csv.xls.csv", header=0)

def split_row(text):
    if ",positive," in text:
        parts = text.split(",positive,", 1)
        return parts[0], "positive", parts[1]
    elif ",negative," in text:
        parts = text.split(",negative,", 1)
        return parts[0], "negative", parts[1]
    else:
        return None, None, None

# Apply split
df[['review', 'sentiment', 'cleaned_review']] = df.iloc[:,0].apply(
    lambda x: pd.Series(split_row(x))
)

# Drop broken column
df = df.drop(columns=[df.columns[0]])

print(df.head())

                                              review sentiment  \
0  One of the other reviewers has mentioned that ...  positive   
1  A wonderful little production. <br /><br />The...  positive   
2  I thought this was a wonderful way to spend ti...  positive   
3  Basically there's a family where a little boy ...  negative   
4  Petter Mattei's "Love in the Time of Money" is...  positive   

                                      cleaned_review  
0  one reviewer mentioned watching 1 oz episode y...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically there family little boy jake think t...  
4  petter matteis love time money visually stunni...  


In [3]:
print(df.columns)

Index(['review', 'sentiment', 'cleaned_review'], dtype='object')


In [4]:
print(df.isnull().sum())

review            0
sentiment         0
cleaned_review    0
dtype: int64


## Tokenization

In [5]:
df['tokens'] = df['cleaned_review'].apply(lambda x: x.split())

In [6]:
print(df[['cleaned_review', 'tokens']].head(3))

                                      cleaned_review  \
0  one reviewer mentioned watching 1 oz episode y...   
1  wonderful little production filming technique ...   
2  thought wonderful way spend time hot summer we...   

                                              tokens  
0  [one, reviewer, mentioned, watching, 1, oz, ep...  
1  [wonderful, little, production, filming, techn...  
2  [thought, wonderful, way, spend, time, hot, su...  


In [7]:
df['tokens'].head(2)

0    [one, reviewer, mentioned, watching, 1, oz, ep...
1    [wonderful, little, production, filming, techn...
Name: tokens, dtype: object

## Normalization

In [8]:
import re

def normalize(tokens):
    clean_tokens = []
    
    for word in tokens:
        word = word.lower()
        
        # remove anything not letters
        word = re.sub(r'[^a-z]', '', word)
        
        # keep words with length >= 3
        if len(word) >= 3:
            clean_tokens.append(word)
    
    return clean_tokens

df['tokens'] = df['tokens'].apply(normalize)

In [9]:
print(df['tokens'].head(2))

0    [one, reviewer, mentioned, watching, episode, ...
1    [wonderful, little, production, filming, techn...
Name: tokens, dtype: object


## remove short words

In [10]:
df['tokens'] = df['tokens'].apply(lambda tokens: [w for w in tokens if len(w) >= 3])

## Stopword Removal

In [11]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Sameh
[nltk_data]     Naiem\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:

stop_words = set(stopwords.words('english'))

custom_stopwords = stop_words.union({
    'one', 'two', 'also', 'would', 'could'
})

def remove_stopwords(tokens):
    return [word for word in tokens if word not in custom_stopwords]

df['tokens'] = df['tokens'].apply(remove_stopwords)

In [13]:
print(df['tokens'].head(2))

0    [reviewer, mentioned, watching, episode, youll...
1    [wonderful, little, production, filming, techn...
Name: tokens, dtype: object


## Lemmatization (token level)

In [14]:
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def lemmatize(tokens):
    return [lemmatizer.lemmatize(word, pos='v') for word in tokens]

df['tokens'] = df['tokens'].apply(lemmatize)

[nltk_data] Downloading package wordnet to C:\Users\Sameh
[nltk_data]     Naiem\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [15]:
print(df['tokens'].head(2))

0    [reviewer, mention, watch, episode, youll, hoo...
1    [wonderful, little, production, film, techniqu...
Name: tokens, dtype: object


## Reconstruct Text

In [16]:
df['processed_text'] = df['tokens'].apply(lambda x: " ".join(x))

In [17]:
print(df[['cleaned_review', 'processed_text']].head(3))

                                      cleaned_review  \
0  one reviewer mentioned watching 1 oz episode y...   
1  wonderful little production filming technique ...   
2  thought wonderful way spend time hot summer we...   

                                      processed_text  
0  reviewer mention watch episode youll hook righ...  
1  wonderful little production film technique una...  
2  think wonderful way spend time hot summer week...  


## Quality Check

In [18]:
print(df[['cleaned_review', 'processed_text']].sample(5))

                                          cleaned_review  \
4353   bit preachy topic progress saving grace mankin...   
37507  perhaps absolute greatest entry hammer house h...   
43920  brutally straightforward tale murder capital p...   
44784  tale crypt house start christmas eve elizabeth...   
1991   seen many bad review supervivientes de los and...   

                                          processed_text  
4353   bite preachy topic progress save grace mankind...  
37507  perhaps absolute greatest entry hammer house h...  
43920  brutally straightforward tale murder capital p...  
44784  tale crypt house start christmas eve elizabeth...  
1991   see many bad review supervivientes los andes f...  


## Saving Final File

In [19]:
final_df = df[['processed_text', 'sentiment']]

final_df.to_csv("processed_text.csv", index=False)